# Company B Bronze Ingestion

This notebook incrementally ingests legacy Company B source files
from Amazon S3 into Bronze Delta tables using Databricks Auto Loader.

The Bronze layer preserves source values without applying business
cleaning or standardization.

In [0]:
from pyspark.sql.functions import *

### Ingesting the Customer Data into Bronze

**Setting the Path**

In [0]:
CUSTOMERS_SOURCE_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "company_b/raw/customers/"
)

CUSTOMERS_SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_b/customers/"
)

CUSTOMERS_CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_b/customers/"
)

CUSTOMERS_TARGET_TABLE = (
    "workspace.bronze.company_b_customers"
)

**Reading using Auto Loader**

In [0]:
customers_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "csv",
    )
    .option(
        "header",
        "true",
    )
    .option(
        "cloudFiles.schemaLocation",
        CUSTOMERS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "false",
    )
    .load(
        CUSTOMERS_SOURCE_PATH
    )
)

**Adding the metadata**

In [0]:
customers_bronze_df = (
    customers_raw_df
    .withColumn(
        "_source_system",
        lit("company_b"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col(
            "_metadata.file_modification_time"
        ),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Customer Data to Bronze Delta table**

In [0]:
customers_query = (
    customers_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CUSTOMERS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        CUSTOMERS_TARGET_TABLE
    )
)

customers_query.awaitTermination()

print(
    "Company B customers Bronze ingestion completed."
)

**Validate the bronze table**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_customer_count
FROM workspace.bronze.company_b_customers
""").show()

In [0]:
spark.sql("""
SELECT
    _source_system,
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_b_customers
GROUP BY
    _source_system,
    _source_file
""").show(truncate=False)

In [0]:
display(
    spark.table(
        "workspace.bronze.company_b_customers"
    )
)

### Ingesting Products data into Bronze

**Setting the path**

In [0]:
PRODUCTS_SOURCE_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "company_b/raw/products/"
)

PRODUCTS_SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_b/products/"
)

PRODUCTS_CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_b/products/"
)

PRODUCTS_TARGET_TABLE = (
    "workspace.bronze.company_b_products"
)

**Reading using Autoloader**

In [0]:
products_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        PRODUCTS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "false",
    )
    .load(PRODUCTS_SOURCE_PATH)
)

**Adding the metadata**

In [0]:
products_bronze_df = (
    products_raw_df
    .withColumn(
        "_source_system",
        lit("company_b"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Products Data to Bronze Delta table**

In [0]:
products_query = (
    products_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        PRODUCTS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        PRODUCTS_TARGET_TABLE
    )
)

products_query.awaitTermination()

print(
    "Company B products Bronze ingestion completed."
)

**Validate the Products**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_product_count
FROM workspace.bronze.company_b_products
""").show()

In [0]:
spark.sql("""
SELECT
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_b_products
GROUP BY _source_file
""").show(truncate=False)

### Ingesting Orders data into Bronze

**Setting the path**

In [0]:
ORDERS_SOURCE_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "company_b/raw/orders/"
)

ORDERS_SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_b/orders/"
)

ORDERS_CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_b/orders/"
)

ORDERS_TARGET_TABLE = (
    "workspace.bronze.company_b_orders"
)

**Reading with Autoloader**

In [0]:
orders_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        ORDERS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "false",
    )
    .load(ORDERS_SOURCE_PATH)
)

**Adding the metadata**

In [0]:
orders_bronze_df = (
    orders_raw_df
    .withColumn(
        "_source_system",
        lit("company_b"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Orders Data to Bronze Delta table**

In [0]:
orders_query = (
    orders_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        ORDERS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        ORDERS_TARGET_TABLE
    )
)

orders_query.awaitTermination()

print(
    "Company B orders Bronze ingestion completed."
)

**Validate**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_order_count
FROM workspace.bronze.company_b_orders
""").show()

In [0]:
spark.sql("""
SELECT
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_b_orders
GROUP BY _source_file
ORDER BY _source_file
""").show(truncate=False)

### Ingesting Orders_Items data into Bronze

**Setting the path**

In [0]:
ORDER_ITEMS_SOURCE_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "company_b/raw/order_items/"
)

ORDER_ITEMS_SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_b/order_items/"
)

ORDER_ITEMS_CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_b/order_items/"
)

ORDER_ITEMS_TARGET_TABLE = (
    "workspace.bronze.company_b_order_items"
)

**Reading with Autoloader**

In [0]:
order_items_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        ORDER_ITEMS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "false",
    )
    .load(ORDER_ITEMS_SOURCE_PATH)
)

**Adding the metadata**

In [0]:
order_items_bronze_df = (
    order_items_raw_df
    .withColumn(
        "_source_system",
        lit("company_b"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Orders_items Data to Bronze Delta Table**

In [0]:
order_items_query = (
    order_items_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        ORDER_ITEMS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        ORDER_ITEMS_TARGET_TABLE
    )
)

order_items_query.awaitTermination()

print(
    "Company B order items Bronze ingestion completed."
)

**Validate**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_order_item_count
FROM workspace.bronze.company_b_order_items
""").show()

In [0]:
spark.sql("""
SELECT
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_b_order_items
GROUP BY _source_file
ORDER BY _source_file
""").show(truncate=False)